## Model Quantization and Compilation with Vitis AI

In this notebook we take the trained FP32 model (from the previous notebook) and apply Vitis AI quantization to obtain an INT8 model.

We will cover:
1. **Post-Training Quantization (PTQ)**: quantize a trained model using a small calibration dataset.
2. **Compilation** to generate an `.xmodel` file for execution on the FPGA DPU.
3. (Optional) **Quantization-Aware Training (QAT)**: fine-tune the model while simulating quantization during training.

> Important: This notebook must be run inside the Vitis AI Docker environment, because the quantizer and compiler tools are available there!

Before running this notebook, you should have:
- Completed the training notebook and saved a model in `./results/` (e.g., `MLP.h5` or `CNN.h5`)

In [ ]:
#Import some basic required libraries

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import random

seed = 42
os.environ["PYTHONHASHSEED"] = str(seed)
random.seed(seed)
np.random.seed(seed)
tf.keras.utils.set_random_seed(seed)

### 1. Dataset Loading (same preprocessing as training)
We reload MNIST using the same preprocessing parameters used during training. We also split the training set into training set and validation set (`val_size = 10000`)

In [ ]:
from src.dataset import load_mnist, split_train_val

(x_train, y_train), (x_test, y_test) = load_mnist(
    limit_test=5000,
    normalize=True,
    add_channel_dim=True,
    one_hot=True
)

x_train, x_val, y_train, y_val = split_train_val(x_train, y_train, val_size=10000)
print(
    "Dataset shapes:\n"
    f"Train: {(x_train.shape, y_train.shape)}\n"
    f"Validation: {(x_val.shape, y_val.shape)}\n"
    f"Test: {(x_test.shape, y_test.shape)}"
)

### 2. Load the trained FP32 model
We load the trained model saved in `./results/`.
This FP32 model will be used as input for Vitis AI quantization.

In [ ]:
#Load the model to be quantized and verify that it is correct 
model_name = "MLP"  
model_path = f"./results/{model_name}.h5"
model = tf.keras.models.load_model(model_path)
model.summary()

### 3. Post-Training Quantization (PTQ)

PTQ converts a trained FP32 model into an INT8 model without re-training. 
Key idea: *calibration*
- A small calibration dataset is passed through the network
- The quantizer observes the range of activations and weights in each layer.
- Based on these statistics, the quantizer determines the scaling factors needed to map floating-point values to INT8.

This approach is very fast and simple because it does not require additional training.

However, for more complex networks, the model accuracy may decrease slightly after quantization.

First of all, let's import the Vitis AI quantization module:

In [ ]:
from tensorflow_model_optimization.quantization.keras import vitis_quantize

This module provides the tools required to quantize TensorFlow/Keras models using the Vitis AI framework.

In particular, it contains the class `VitisQuantizer`, which is the main interface used to apply quantization.

The `VitisQuantizer` takes a trained FP32 model and prepares it for quantization by applying a specific *quantization strategy*. In Vitis AI, several quantization strategies are available:

- `pof2s` (Power-Of-Two Scaling) – This is the **default** strategy designed for the DPU.   
  The scaling factors used to map floating-point values to INT8 are constrained to powers of two.  
  This allows the quantization scaling to be implemented in hardware using bit-shift operations instead of multiplications, which makes inference more efficient on the FPGA.

- `pof2s_tqt` (Power-Of-Two Scaling with Trained Quantization Thresholds) – Similar to `pof2s`, but the quantization thresholds are learned during training. This strategy is typically used with QAT to improve accuracy after quantization.

- `fs` (Floating Scale) and `fsx` (Extended Floating Scale) – Uses floating-point scaling factors instead of powers of two.  
  This allows more flexible quantization ranges, but is generally less hardware-efficient, and indeed is not supported by the DPU.
  

In this tutorial we use the default `pof2s` strategy, which is optimized for deployment on the DPU architecture.

Depending on the method used, `VitisQuantizer` can:

- perform **PTQ** using `quantize_model()`
- enable **QAT** using `get_qat_model()`

In this section we apply **PTQ**.

In [ ]:
#Inizialize Vitis AI quantizer 
quantizer = vitis_quantize.VitisQuantizer(
    model,
    quantize_strategy="pof2s",        # default
    custom_quantize_strategy=None,    # optionally, you can set your own quantization strategy (not supported by DPU) 
    custom_objects={}
)

In [ ]:
PTQ_model= quantizer.quantize_model(
    calib_dataset = x_train[0:50])

**Note.** Although we do not explicitly pass other arguments to the function  `quantize_model()`, the function internally accepts additional keyword arguments (`**kwargs`) that allow the user to define custom configurations of the quantization strategy. For more information about the quantization parameters and available configuration options, you can refer to the official Vitis AI documentation: https://docs.amd.com/r/en-US/ug1414-vitis-ai/vitis_quantize.VitisQuantizer. In this tutorial we use the default built-in `pof2s` quantization strategy, so we do not need to specify additional parameters. This corresponds to an **signed INT8 symmetric quantization scheme** for inputs, weights, activations, and biases.

At this point, the PTQ process is complete. 
After quantization, the model must be compiled in Keras using `PTQ_model.compile()` before it can be evaluated on the test dataset. Finally, the model must be saved, since the saved quantized model will be used in the next step to generate the FPGA-compatible `.xmodel` file. 

In [ ]:
#Compile the quantized model
PTQ_model.compile(loss=tf.keras.losses.CategoricalCrossentropy(from_logits=False), metrics=["accuracy"])
PTQ_model.summary()

In [ ]:
#Calculate the performance of the quantized model on the test dataset, using the evaluation metrics in ./src/evaluation.py
from src.evaluation import compute_metrics, plot_confusion_matrix
# Predictions
y_pred=PTQ_model.predict(x_test)
y_pred_classes = np.argmax(y_pred, axis=1) 

# True labels
y_true = np.argmax(y_test, axis=1) if np.ndim(y_test) > 1 else y_test

# Metrics
metrics = compute_metrics(y_true, y_pred_classes, compute_f1=False, compute_recall=False)

print("\nEvaluation metrics on test dataset:")
for name, value in metrics.items():
    print(f"{name}: {value:.5f}")

Classes=[str(i) for i in range(10)]

# Confusion matrix
plt.figure(figsize=(7,6))
plot_confusion_matrix(
    y_true,
    y_pred_classes,
    classes=Classes,
    title="Confusion Matrix - Test dataset",
    normalize=True
)

In [ ]:
#Save the quantized model

# Create folder if it does not exist
os.makedirs("results", exist_ok=True)
PTQ_model.save(f'./results/{model_name}_quantized_PTQ.h5')

The file `src/model_evaluation.py` contains also a function to compare the size of the model before and after the quantization: 

In [ ]:
from src.evaluation import compute_model_size_reduction

compute_model_size_reduction(
    f"./results/{model_name}.h5",
    f"./results/{model_name}_quantized_PTQ.h5"
);

### Question 

**Q4.** How does the model accuracy change compared to the model disk size after quantization?

### 4. Vitis AI Compiler

Now we are ready to transform the model into a format that can be interpreted and executed by the DPU! We now run the Vitis AI compiler (VAI_C). For TensorFlow 2 models, the compiler front-end is: `vai_c_tensorflow2`. The compiler takes the INT8 quantized model and maps the neural network operations to the DPU instruction set, by generating an executable model in the `.xmodel` format. 

What the compiler needs:
- the quantized model (`.h5`)
- the target DPU architecture description (`arch*.json`)
- an output folder and a network name

The `arch.json` file describes the target DPU configuration (instruction set, supported ops, compute resources, etc.).  
The compiler uses it to generate an `.xmodel` that matches the exact DPU implemented on the board. For pre-built MPSoC DPU platforms, Vitis AI already provides a set of ready-to-use architecture files inside the Docker container under: `/opt/vitis_ai/compiler/arch/DPUCZDX8G/`. 

Here, we will use the architecture file corresponding to the **Kria KV260 board**, which features a Zynq UltraScale+ MPSoC. 

The compilation command is as follows:

In [ ]:
import subprocess
# Construct the command for `vai_c_tensorflow2`
vai_c_command = [
    'vai_c_tensorflow2',
    '--model', f'./results/{model_name}_quantized_PTQ.h5',  #Quantized model to be converted in the FPGA-compliant format 
    '--arch', '/opt/vitis_ai/compiler/arch/DPUCZDX8G/KV260/arch.json',  #FPGA architecture 
    '--output_dir', f'./compilation',   #Folder in which to save the FPGA-compliant format model (.xmodel) 
    '--net_name', f'{model_name}'  #Name with which to save the FPGA-compliant format model (.xmodel) 
]

# Execute the command
subprocess.run(vai_c_command, check=True)

If the compilation completes successfully, the Vitis AI compiler generates the DPU-executable `.xmodel` file and prints the message `Compile done.` together with the path where the compiled model has been saved.

### Question 

**Q5.** Try changing the quantization strategy in: `quantizer = vitis_quantize.VitisQuantizer(model, quantize_strategy="pof2s")`. For example, replace `pof2s` with `fs`. Then repeat PTQ and try to compile the quantized model using the Vitis AI compiler. Do you notice anything unusual?

### 5. (Optional) A 2nd method for quantization: QUANTIZATION AWARE-TRAINING (QAT)

In QAT, the quantization effects are simulated during the training process. In the Vitis AI framework, QAT can be enabled using the `VitisQuantizer` class through the function:

`get_qat_model()`

This function converts the original FP32 model into a **quantization-aware model**, where special layers simulate the behavior of the INT8 arithmetic used by the DPU.

The typical QAT workflow consists of several steps:
1. Initialize the Vitis AI quantizer by loading the trained FP32 model: `quantizer = vitis_quantize.VitisQuantizer(model, ...)`

2. Create the QAT model: `qat_model = quantizer.get_qat_model(...)`

3. Train (or fine-tune) the QAT model

4. Convert the trained QAT model into a deployable quantized model `deploy_model = vitis_quantize.VitisQuantizer.get_deploy_model(...)`

5. Evaluate the quantized model on the test dataset

6. Save the quantized model, which will then be compiled using the Vitis AI compiler to generate the `.xmodel` file executable on the DPU.

In [ ]:
#1 and 2. Inizialize the quantizer and create the QAT model 
quantizer = vitis_quantize.VitisQuantizer(model, quantize_strategy="pof2s_tqt")
qat_model = quantizer.get_qat_model(
    calib_dataset= x_train[0:50])

In [ ]:
#3. Start the quantization aware training process. We will train the model for 3 epochs. 
import time

#Before training, you should compile the model 
qat_model.compile(optimizer='adam',
                  loss=tf.keras.losses.CategoricalCrossentropy(from_logits=False),
                  metrics=['accuracy'])

learn_rate=0.00001
batch_size = 256
epochs=3
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath=f"./results/{model_name}_qat_best_weights.h5",
    save_best_only=True,
    monitor="val_loss",
    save_weights_only=True
)


train_batches = (tf.data.Dataset.from_tensor_slices((x_train, y_train))
                 .shuffle(x_train.shape[0])
                 .batch(batch_size)
                 .prefetch(1))

val_batches = (tf.data.Dataset.from_tensor_slices((x_val, y_val))
               .batch(batch_size)
               .prefetch(1))

print("Fitting the model...")
t = time.time()

results = qat_model.fit(
    train_batches,
    validation_data=val_batches,
    epochs=epochs,
    callbacks=[checkpoint]
)

In [ ]:
#4. Deploy the quantized model 
quantized_model_QAT = vitis_quantize.VitisQuantizer.get_deploy_model(qat_model)
quantized_model_QAT.summary()

In [ ]:
#5. Compile the quantized model 
quantized_model_QAT.compile(loss=tf.keras.losses.CategoricalCrossentropy(from_logits=False), metrics=["accuracy"])

#Calculate the performance of the quantized model on the test dataset 
from src.evaluation import compute_metrics, plot_confusion_matrix
# Predictions
y_pred=quantized_model_QAT.predict(x_test)
y_pred_classes = np.argmax(y_pred, axis=1) 

# True labels
y_true = np.argmax(y_test, axis=1) if np.ndim(y_test) > 1 else y_test

# Metrics
metrics = compute_metrics(y_true, y_pred_classes, compute_f1=False, compute_recall=False)

print("\nEvaluation metrics on test dataset:")
for name, value in metrics.items():
    print(f"{name}: {value:.5f}")

Classes=[str(i) for i in range(10)]

# Confusion matrix
plt.figure(figsize=(7,6))
plot_confusion_matrix(
    y_true,
    y_pred_classes,
    classes=Classes,
    title="Confusion Matrix - Test dataset",
    normalize=True
)

In [ ]:
#6. Save the quantized model 
os.makedirs("results", exist_ok=True)
quantized_model_QAT.save(f'./results/{model_name}_quantized_QAT.h5')

If you want also compile the quantized model for execution on the DPU, run: 

```bash
import subprocess
# Construct the command for `vai_c_tensorflow2`
vai_c_command = [
    'vai_c_tensorflow2',
    '--model', f'./results/{model_name}_quantized_QAT.h5',  #Quantized model to be converted in the FPGA-compliant format 
    '--arch', '/opt/vitis_ai/compiler/arch/DPUCZDX8G/KV260/arch.json',  #FPGA architecture 
    '--output_dir', f'./compilation',   #Folder in which to save the FPGA-compliant format model (.xmodel) 
    '--net_name', f'{model_name}'  #Name with which to save the FPGA-compliant format model (.xmodel) 
]

# Execute the command
subprocess.run(vai_c_command, check=True)
```

#### Exercise.
Try to repeat the same quantization and compilation steps for the CNN model trained in the first notebook.

Starting from **Step 2 (Load the trained model)**, simply replace the model name with:

`model_name = "CNN"`

Then repeat the same workflow.